# NIH Chest X-Ray Dataset: Preprocessing & Filtering Pipeline
This notebook outlines the exact steps taken to process the original [NIH Chest X-Ray Dataset](https://www.kaggle.com/datasets/nih-chest-xrays/data/data).
The original dataset contains 112,120 high-resolution X-ray images with 14 different disease labels. 

**Our Presentation Goal:**
1. **Filter** the dataset to only include our 4 target conditions: **Pneumonia, Effusion, Cardiomegaly, and Pneumothorax**, plus **No Finding**.
2. **Resize** all high-resolution images (1024x1024) to a lightweight size (224x224) to dramatically speed up model training and reduce storage requirements.
3. **Generate** the final `resized_labels.csv` to be used for our custom DenseNet-121 model training.

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt
import shutil

# Paths to original Kaggle dataset (assumed to be downloaded locally)
ORIGINAL_IMAGES_DIR = './images/'
ORIGINAL_CSV_PATH = 'Data_Entry_2017.csv'

# Paths for output
OUTPUT_DIR = './resized_images/'
OUTPUT_CSV = 'resized_labels.csv'
TARGET_SIZE = (224, 224)

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load and Analyze the Original Kaggle Data
The original dataset uses a `|` separated string for multi-label classification (e.g., `Atelectasis|Effusion`).

In [ ]:
# Load the original metadata
df = pd.read_csv(ORIGINAL_CSV_PATH)

print(f"Total original images: {len(df)}")
df.head()

## 2. Filter for Our Specific Target Conditions
We will map the original `Finding Labels` column into strict one-hot encoded columns for our specific 4 conditions and `No Finding`. We will intentionally drop images that only contain irrelevant diseases like 'Mass' or 'Fibrosis' to focus our model's learning capabilities on what matters to our specific application.

In [ ]:
TARGET_DISEASES = ['Pneumonia', 'Effusion', 'Cardiomegaly', 'Pneumothorax']

# Create one-hot encoded columns for our targets
for disease in TARGET_DISEASES:
    df[disease] = df['Finding Labels'].apply(lambda x: 1 if disease in x else 0)

# Extract 'No Finding'
df['No Finding'] = df['Finding Labels'].apply(lambda x: 1 if 'No Finding' in x else 0)

# Filter the dataframe to only include rows where at least one of our 5 targets is present (1)
df['Keep'] = df[TARGET_DISEASES + ['No Finding']].max(axis=1)
filtered_df = df[df['Keep'] == 1].copy()

print(f"Images after filtering for target diseases: {len(filtered_df)}")

# Display the class distribution
print("\nClass Distribution for our Custom Dataset:")
for col in TARGET_DISEASES + ['No Finding']:
    print(f"{col}: {filtered_df[col].sum()}")

## 3. Image Resizing and Transfer Pipeline
To make training mathematically efficient for the deep learning models, we will read the images from the raw Kaggle folders, resize them down using OpenCV, and save them to our new targeted `resized_images/` folder.

In [ ]:
def process_and_resize_images(dataframe, source_dir, dest_dir, target_size):
    processed_count = 0
    missing_count = 0
    
    # We only need the Image Index and our target columns for the final CSV
    final_data = []
    
    print("Starting image preprocessing and resizing pipeline...")
    for index, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
        img_name = row['Image Index']
        source_path = os.path.join(source_dir, img_name)
        dest_path = os.path.join(dest_dir, img_name)
        
        # Check if the image exists in the original folder
        if os.path.exists(source_path):
            try:
                # Read image in grayscale to remove unnecessary color channels
                img = cv2.imread(source_path, cv2.IMREAD_GRAYSCALE)
                
                if img is not None:
                    # Resize the image to 224x224 using Area Interpolation
                    resized_img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
                    
                    # Save to new folder
                    cv2.imwrite(dest_path, resized_img)
                    processed_count += 1
                    
                    # Append data to our final array
                    final_data.append({
                        'Image_Index': img_name,
                        'Pneumonia': row['Pneumonia'],
                        'Effusion': row['Effusion'],
                        'Cardiomegaly': row['Cardiomegaly'],
                        'Pneumothorax': row['Pneumothorax'],
                        'No_Finding': row['No Finding']
                    })
            except Exception as e:
                print(f"Error processing {img_name}: {e}")
        else:
            missing_count += 1
            
    print(f"\nPipeline complete!")
    print(f"Successfully resized: {processed_count}")
    print(f"Missing images: {missing_count}")
    
    return pd.DataFrame(final_data)

# UNCOMMENT TO RUN THE FULL RESIZING PIPELINE
# final_df = process_and_resize_images(filtered_df, ORIGINAL_IMAGES_DIR, OUTPUT_DIR, TARGET_SIZE)

## 4. Save Final CSV and Visualize Results
Save the filtered dataset to match the structure expected by our C# backend and Python training scripts, and visualize a sample of our final dataset.

In [ ]:
# UNCOMMENT TO SAVE
# final_df.to_csv(OUTPUT_CSV, index=False)
# print(f"Final dataset labels saved to {OUTPUT_CSV}")

# Visualize a sample from our newly created dataset
def visualize_sample(img_dir, df, num_samples=3):
    sample_df = df.sample(num_samples)
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    
    for idx, (index, row) in enumerate(sample_df.iterrows()):
        img_path = os.path.join(img_dir, row['Image_Index'])
        if os.path.exists(img_path):
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            axes[idx].imshow(img, cmap='gray')
            
            # Get positive labels for display
            labels = []
            for d in TARGET_DISEASES + ['No Finding']:
                if d == 'No Finding' and row['No_Finding'] == 1:
                    labels.append(d)
                elif d != 'No Finding' and row[d] == 1:
                    labels.append(d)
                    
            axes[idx].set_title("\n".join(labels))
            axes[idx].axis('off')
            
    plt.tight_layout()
    plt.show()

# visualize_sample(OUTPUT_DIR, final_df)